In [1]:
from typing import Any, Dict, List

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import HandoffTermination, TextMentionTermination
from autogen_agentchat.messages import HandoffMessage
from autogen_agentchat.teams import Swarm
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

In [ ]:
# tool
def refund_flight (flight_PNR:str) ->str:
    return f"Refunded Flight with PNR {flight_PNR}"

In [3]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

In [8]:
model_client = OpenAIChatCompletionClient(model='gpt-4o',api_key=api_key)

In [9]:
# test
my_assistant = AssistantAgent(name='Assistant',model_client=model_client)
result = await my_assistant.run(task='Who are you')
print(result)

messages=[TextMessage(id='86a6454e-901d-4d21-8795-b1437be6f2fd', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 28, 12, 49, 8, 762199, tzinfo=datetime.timezone.utc), content='Who are you', type='TextMessage'), TextMessage(id='fcadd46c-e299-459a-a97d-419e4dc20408', source='Assistant', models_usage=RequestUsage(prompt_tokens=39, completion_tokens=39), metadata={}, created_at=datetime.datetime(2025, 7, 28, 12, 49, 10, 896615, tzinfo=datetime.timezone.utc), content='I am an AI assistant designed to help you solve problems and answer questions. My purpose is to assist you by providing information, performing tasks, and facilitating interactions. How can I assist you today?', type='TextMessage')] stop_reason=None


In [10]:
travel_agent = AssistantAgent(
    "travel_agent",
    model_client=model_client,
    handoffs=["flights_refunder", "user"],
    system_message="""You are a travel agent.
    The flights_refunder is in charge of refunding flights.
    If you need information from the user, you must first send your message, then you can handoff to the user.
    Use TERMINATE when the travel planning is complete.""",
)

In [11]:
flights_refunder = AssistantAgent(
    "flights_refunder",
    model_client=model_client,
    handoffs=["travel_agent", "user"],
    tools=[refund_flight],
    system_message="""You are an agent specialized in refunding flights.
    You only need flight PNR numbers to refund a flight.
    You have the ability to refund a flight using the refund_flight tool.
    If you need information from the user, you must first send your message, then you can handoff to the user.
    When the transaction is complete, handoff to the travel agent to finalize.""",
)

In [12]:
termination = HandoffTermination(target="user") | TextMentionTermination("TERMINATE")
team = Swarm([travel_agent, flights_refunder], termination_condition=termination)

In [13]:
task = ' I want to refund my flight'
async def run_team_stream() -> None:

    task_result = await Console(team.run_stream(task = task))
    last_message = task_result.messages[-1]


    while ( isinstance(last_message,HandoffMessage) and last_message.target == 'user'):

        user_ka_message = input("User : ")

        task_result = await Console(team.run_stream(task = HandoffMessage(source='user',target=last_message.source,content=user_ka_message)))

        last_message = task_result.messages[-1]

await run_team_stream()

---------- TextMessage (user) ----------
 I want to refund my flight
---------- ThoughtEvent (travel_agent) ----------
I can assist you with that. I'll need to transfer you to the flights_refunder who specializes in handling flight refunds. Please hold on for a moment.
---------- ToolCallRequestEvent (travel_agent) ----------
[FunctionCall(id='call_0slHkC6quFrTeOraNStIKp0h', arguments='{}', name='transfer_to_flights_refunder')]
---------- ToolCallExecutionEvent (travel_agent) ----------
[FunctionExecutionResult(content='Transferred to flights_refunder, adopting the role of flights_refunder immediately.', name='transfer_to_flights_refunder', call_id='call_0slHkC6quFrTeOraNStIKp0h', is_error=False)]
---------- HandoffMessage (travel_agent) ----------
Transferred to flights_refunder, adopting the role of flights_refunder immediately.
---------- TextMessage (flights_refunder) ----------
To process your flight refund, I need your flight PNR number. Could you please provide that?
---------- 